<a href="https://colab.research.google.com/github/Balachandar-Ganesan/GraphRAG/blob/main/BuildYourOwnLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget https://raw.githubusercontent.com/Balachandar-Ganesan/DeepLearning/refs/heads/main/TinyStories-KeyWords-18.csv



In [ ]:
!pip install tiktoken torch pandas


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import tiktoken

# 1. SETUP DATA: Using your described 3-word and 30-word structure
custom_headers = ['title', 'content']
df = pd.read_csv("/content/TinyStories-KeyWords-18.csv",names=custom_headers, header=None)

# 2. TOKENIZATION: Using tiktoken (GPT-2 encoding)
# changed to 0299k_base
enc = tiktoken.get_encoding("o200k_base")

def tokenize_dataframe(df):
    all_tokens = []
    for _, row in df.iterrows():
        # Combine columns with a separator
        combined_text = f"{row['title']} <|endoftext|> {row['content']}"
        tokens = enc.encode(combined_text, allowed_special={'<|endoftext|>'})
        all_tokens.extend(tokens)
    return all_tokens

tokens = tokenize_dataframe(df)

# 3. DATASET CLASS: Preparing sequences for the LLM
class TextDataset(Dataset):
    def __init__(self, tokens, seq_length=32):
        self.tokens = torch.tensor(tokens, dtype=torch.long)
        self.seq_length = seq_length

    def __len__(self):
        return len(self.tokens) - self.seq_length

    def __getitem__(self, idx):
        # x is the input sequence, y is the target (next token)
        x = self.tokens[idx : idx + self.seq_length]
        y = self.tokens[idx + 1 : idx + self.seq_length + 1]
        return x, y

dataset = TextDataset(tokens)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

# 4. MODEL ARCHITECTURE: A Mini-Transformer
class MiniLLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, n_heads=4, n_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoding = nn.Parameter(torch.zeros(1, 1024, embed_dim))

        # Transformer Layers
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=n_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Output Head
        self.fc_out = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        x = self.embedding(x) + self.pos_encoding[:, :seq_len, :]
        x = self.transformer(x)
        return self.fc_out(x)

# Initialize Model, Optimizer, and Loss
model = MiniLLM(vocab_size=enc.n_vocab)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

# 5. TRAINING LOOP
print("Starting training...")
model.train()
for epoch in range(10): # 10 epochs for demonstration
    total_loss = 0
    for x, y in loader:
        optimizer.zero_grad()
        logits = model(x)

        # Reshape for CrossEntropyLoss
        loss = criterion(logits.view(-1, enc.n_vocab), y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(loader):.4f}")

print("Training Complete!")


Starting training...
Epoch 1 | Loss: 8.5020
Epoch 2 | Loss: 3.9969
Epoch 3 | Loss: 2.1940
Epoch 4 | Loss: 1.2074
Epoch 5 | Loss: 0.7096
Epoch 6 | Loss: 0.4617
Epoch 7 | Loss: 0.3390
Epoch 8 | Loss: 0.2734
Epoch 9 | Loss: 0.2354


In [45]:
def generate_text(model, start_text, max_new_tokens=30, temperature=0.5):
    model.eval()
    # Convert prompt to tokens
    tokens = enc.encode(start_text)
    input_tensor = torch.tensor([tokens], dtype=torch.long)

    for _ in range(max_new_tokens):
        with torch.no_grad():
            # Get predictions for the next token
            logits = model(input_tensor)
            # Focus only on the last token's output
            next_token_logits = logits[0, -1, :] / temperature

            # Use Softmax to get probabilities and sample
            probs = torch.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()

            # Append to the sequence
            tokens.append(next_token)
            input_tensor = torch.cat([input_tensor, torch.tensor([[next_token]])], dim=1)

            # Stop if the model generates the "end of text" token
            if next_token == enc.eot_token:
                break

    return enc.decode(tokens)




In [62]:
# --- Execution ---
# Ensure the model is trained first, then run:
prompt = "Villager Murder Conspiracy"
generated_output = generate_text(model, prompt, max_new_tokens=30)
print(f"\nPrompt: {prompt}")
print(f"Generated: {generated_output}")


Prompt: Villager Murder Conspiracy
Generated: Villager Murder Conspiracy now.Musician dancer competition of artistic skill and fragile ego between a graceful dancer competition <|endoftext|>
